# GPT OSS 120B — RLVR with Tool-Integrated Reasoning

Train `openai/gpt-oss-120b` via **RLVR** with **Python tool use** using the Tinker `tinker_cookbook.rl` framework.

**Architecture:**
- **Multi-turn environment** via `MessageEnv` + `EnvFromMessageEnv`
- Model generates reasoning → calls `python` tool → gets execution output → continues → produces `\boxed{}` answer
- **Local Jupyter kernel** for code execution during rollouts (no Modal required)
- GRPO with importance sampling loss, group_size=4
- `GptOssRenderer` for Harmony format (handles `<|call|>`, `<|return|>` routing)

**Reward:**
- `+1.0` for correct `\boxed{}` answer matching ground truth
- `-1.0` for wrong answer or no `\boxed{}`
- `-0.1` for context overflow (ran out of tokens)
- `-1.0` for parse errors

In [1]:
# ============================================================
# Cell 1: Configuration (UNCHANGED except group_size note)
# ============================================================

class RLConfig:
    model_name = "openai/gpt-oss-120b"
    lora_rank = 32
    learning_rate = 1e-5          # Lower LR for RL (vs 2e-4 for SFT)
    max_tokens = 4096             # Max tokens PER generation step (per turn)
    max_trajectory_tokens = 24576 # Total token budget for entire multi-turn episode
    batch_size = 4               # Problems per training step
    group_size = 8                # Completions per problem (GRPO)
    max_steps = 100               # Total training iterations
    temperature = 1.0             # Sampling temperature for rollouts
    loss_fn = "importance_sampling"  # Standard for GRPO
    eval_every = 20               # Evaluate every N steps
    save_every = 25               # Checkpoint every N steps
    log_path = "/kaggle/working/rl_logs"
    
    # Tool execution
    max_tool_iterations = 15      # Max tool calls per episode
    python_timeout = 30.0         # Timeout for each Python execution (seconds)
    
    # Dataset path on Kaggle
    dataset_path = "/kaggle/input/datasets/nahidhossainredom/rlvr-dataset/rlvr_dataset.csv"
    
    # Optional: warm-start from SFT checkpoint
    load_checkpoint_path = None  # e.g. "tinker://SESSION_ID:train:0/sampler_weights/NAME"

print("RLVR + Tool Use Config:")
for k, v in vars(RLConfig).items():
    if not k.startswith('_'):
        print(f"  {k}: {v}")


RLVR + Tool Use Config:
  model_name: openai/gpt-oss-120b
  lora_rank: 32
  learning_rate: 1e-05
  max_tokens: 4096
  max_trajectory_tokens: 24576
  batch_size: 4
  group_size: 8
  max_steps: 100
  temperature: 1.0
  loss_fn: importance_sampling
  eval_every: 20
  save_every: 25
  log_path: /kaggle/working/rl_logs
  max_tool_iterations: 15
  python_timeout: 30.0
  dataset_path: /kaggle/input/datasets/nahidhossainredom/rlvr-dataset/rlvr_dataset.csv
  load_checkpoint_path: None


In [2]:
# ============================================================
# Cell 2: Install Dependencies
# ============================================================

!pip install -q tinker tinker-cookbook

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.0/187.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 850.2/850.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.9 MB/s eta 0:00:00


In [3]:
# ============================================================
# Cell 3: Imports & API Key (UNCHANGED)
# ============================================================

import os, re, math, logging, asyncio, threading, queue, contextlib, io, traceback
from functools import partial
from collections.abc import Sequence
from dataclasses import dataclass, field
from typing import Literal, cast

import pandas as pd
import chz
import tinker

from tinker_cookbook import renderers, model_info
from tinker_cookbook.tokenizer_utils import get_tokenizer
from tinker_cookbook.rl.types import (
    RLDataset, RLDatasetBuilder, EnvGroupBuilder, Env,
    Trajectory, Metrics, StepResult, Action, ActionExtra,
)
from tinker_cookbook.rl.message_env import (
    MessageEnv, MessageStepResult, EnvFromMessageEnv,
)
from tinker_cookbook.rl import train
from tinker_cookbook.completers import StopCondition
from tinker_cookbook.renderers.base import Message, ToolSpec

# Try to import the cookbook's math grading utilities
try:
    from tinker_cookbook.recipes.math_rl.math_grading import (
        extract_boxed,
        grade_answer,
        run_with_timeout_signal,
    )
    HAS_MATH_GRADING = True
    print("\u2713 Loaded tinker_cookbook math grading (SymPy-based)")
except ImportError:
    HAS_MATH_GRADING = False
    print("\u26a0 Math grading not available, using string matching fallback")

# ---- SET YOUR TINKER API KEY HERE ----
os.environ["TINKER_API_KEY"] = "YOUR_API_KEY_HERE"  # <-- REPLACE THIS!

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("gpt-oss-rlvr-tool")

print(f"Tinker SDK version: {tinker.__version__}")
print(f"API key set: {'TINKER_API_KEY' in os.environ and os.environ['TINKER_API_KEY'] != 'YOUR_API_KEY_HERE'}")

⚠ Math grading not available, using string matching fallback
Tinker SDK version: 0.16.1
API key set: True


In [4]:
# ============================================================
# Cell 4: Load & Inspect Dataset
# ============================================================

df = pd.read_csv(RLConfig.dataset_path)

print(f"Dataset loaded: {len(df)} problems")
print(f"Columns: {list(df.columns)}")
print(f"\nAnswer types:")
print(f"  Numeric (int-like): {df['answer'].apply(lambda x: str(x).lstrip('-').isdigit()).sum()}")
print(f"  Other:             {(~df['answer'].apply(lambda x: str(x).lstrip('-').isdigit())).sum()}")

# Preview
print(f"\n{'='*60}")
for i in range(min(3, len(df))):
    row = df.iloc[i]
    # print(f"\n[Problem {row['id']}]")
    print(f"  Q: {str(row['problem'])[:150]}...")
    print(f"  A: {row['answer']}")

Dataset loaded: 29 problems
Columns: ['problem', 'answer']

Answer types:
  Numeric (int-like): 29
  Other:             0

  Q: Let \( n \ge 2 \) be an integer. Alex writes the numbers \( 1, 2, \ldots, n \) in some order on a circle such that any two neighbors are coprime. For ...
  A: 11
  Q: At a mathematical olympiad, eight problems were given to 30 contestants. The points for each problem are assigned based on the number of contestants w...
  A: 58
  Q: Consider two circles \( \omega_1 \) and \( \omega_2 \) intersecting at points \( X \) and \( Y \). Their common tangent closer to \( X \) touches \( \...
  A: 29


In [5]:
# ============================================================
# Cell 5: Python Execution Tool — Lightweight In-Process Executor
# ============================================================
# [FIX v2] REPLACED JupyterSession + JupyterKernelPool entirely.
#
# WHY THE KERNEL POOL APPROACH FAILS:
# -----------------------------------
# make_envs() acquires ALL kernels for a group EAGERLY and holds them
# for the ENTIRE rollout duration (minutes). With batch_size=2, group_size=8:
#   → 16 kernels grabbed instantly = 100% of pool
#   → If eval, retry, or next batch needs even ONE kernel → pool exhausted
#   → cleanup() only releases kernels AFTER the entire group rollout finishes
#   → If any rollout errors before cleanup() → kernels leak PERMANENTLY
#
# Even reducing batch_size doesn't help because the fundamental problem is:
# HOLDING kernels for minutes while only USING them for milliseconds.
#
# THE FIX: CodeExecutor
# ---------------------
# Uses exec() with a shared namespace dict. Zero external resources.
# - No processes, no ports, no file descriptors, no pool
# - Create one per env for FREE (it's just a Python dict)
# - State persists across calls within an episode (shared namespace)
# - Timeout via daemon thread + join()
# - Supports unlimited concurrency — no pool to exhaust

import json  # Needed for tool call arg parsing


class CodeExecutor:
    """Lightweight in-process Python code executor.
    
    Replaces JupyterSession for RL training. Uses exec() with a persistent
    namespace dict so variables defined in one tool call are available in
    the next (within the same episode).
    
    Properties:
        - Zero startup cost (no subprocess, no ports)
        - Zero persistent resources (no kernel process)
        - Unlimited concurrency (no pool needed)
        - State persists across calls (shared globals dict)
        - Thread-based timeout for infinite loops
    """
    
    _PREAMBLE = (
        "import math\n"
        "import numpy as np\n"
        "import sympy as sp\n"
        "from sympy import *\n"
        "import itertools\n"
        "import collections\n"
        "import warnings; warnings.filterwarnings('ignore')\n"
    )
    
    _MAX_OUTPUT_CHARS = 3000
    
    def __init__(self, timeout: float = 30.0):
        self._timeout = timeout
        self._globals: dict = {"__builtins__": __builtins__}
        # Pre-import math libraries into the namespace
        exec(self._PREAMBLE, self._globals)
    
    def execute(self, code: str, timeout: float | None = None) -> str:
        """Execute Python code and return stdout + errors.
        
        Uses a daemon thread with join(timeout) for timeout handling.
        The namespace persists across calls so variables are carried over.
        """
        effective_timeout = timeout or self._timeout
        
        # Shared result container between main thread and executor thread
        result: dict = {"stdout": "", "error": None, "done": False}
        
        def _run():
            """Execute code in a background thread, capturing stdout."""
            buf = io.StringIO()
            try:
                with contextlib.redirect_stdout(buf):
                    exec(code, self._globals)
                result["stdout"] = buf.getvalue()
            except Exception as e:
                result["stdout"] = buf.getvalue()
                # Format a clean traceback: just the error type and message,
                # without the internal exec() frames
                tb_lines = traceback.format_exception(type(e), e, e.__traceback__)
                # Keep only lines from user code (skip exec/CodeExecutor frames)
                clean_lines = []
                skip = True
                for line in tb_lines:
                    if '<string>' in line or 'exec(' not in line:
                        skip = False
                    if not skip:
                        clean_lines.append(line)
                if not clean_lines:
                    clean_lines = [f"{type(e).__name__}: {e}\n"]
                result["error"] = "".join(clean_lines).strip()
            result["done"] = True
        
        # Run in a daemon thread so it doesn't block if it hangs
        thread = threading.Thread(target=_run, daemon=True)
        thread.start()
        thread.join(timeout=effective_timeout)
        
        if not result["done"]:
            # Thread is still running (infinite loop / very slow computation).
            # Since it's a daemon thread, it won't block process exit.
            # The model learns to avoid this from the timeout feedback.
            return f"[TIMEOUT] Execution exceeded {effective_timeout:.0f}s"
        
        stdout = result["stdout"].strip()
        error = result["error"]
        
        if error:
            output = f"{stdout}\n{error}" if stdout else error
        elif stdout:
            output = stdout
        else:
            output = "[No output. Use print() to see results.]"
        
        # Truncate very long output to avoid blowing up the context window
        if len(output) > self._MAX_OUTPUT_CHARS:
            output = output[:self._MAX_OUTPUT_CHARS] + f"\n... [truncated, {len(output)} total chars]"
        
        return output
    
    def reset(self):
        """Reset the namespace for a new episode."""
        self._globals = {"__builtins__": __builtins__}
        exec(self._PREAMBLE, self._globals)
    
    def close(self):
        """No-op. CodeExecutor holds no external resources."""
        self._globals = {}


# Quick test
print("Testing CodeExecutor...")
_test = CodeExecutor(timeout=5.0)

# Test basic execution
result = _test.execute("print(2 + 2)")
assert "4" in result, f"Basic exec failed: {result}"
print(f"  2+2 = {result.strip()}")

# Test state persistence across calls
_test.execute("x = 42")
result = _test.execute("print(x * 2)")
assert "84" in result, f"State persistence failed: {result}"
print(f"  x*2 = {result.strip()}")

# Test sympy
result = _test.execute("print(isprime(17))")
assert "True" in result, f"SymPy test failed: {result}"
print(f"  isprime(17) = {result.strip()}")

# Test error handling
result = _test.execute("1/0")
assert "ZeroDivision" in result, f"Error handling failed: {result}"
print(f"  1/0 → {result.strip()[:50]}")

# Test timeout
result = _test.execute("while True: pass", timeout=2.0)
assert "TIMEOUT" in result, f"Timeout test failed: {result}"
print(f"  infinite loop → {result.strip()}")

# Test reset
_test.reset()
result = _test.execute("try:\n    print(x)\nexcept NameError:\n    print('RESET OK')")
assert "RESET OK" in result, f"Reset test failed: {result}"
print(f"  after reset → {result.strip()}")

_test.close()
print("\u2713 CodeExecutor works — zero external resources, unlimited concurrency")

Testing CodeExecutor...
  2+2 = 4
  x*2 = 84
  isprime(17) = True
  1/0 → Traceback (most recent call last):
  File "/tmp/ip


In [6]:
# ============================================================
# Cell 6: Answer Grading Utilities (UNCHANGED)
# ============================================================

def extract_boxed_answer(text: str) -> str | None:
    """Extract content from the last \\boxed{...} in text, handling nested braces."""
    key = r"\boxed{"
    idx = text.rfind(key)
    if idx < 0:
        return None
    i = idx + len(key)
    depth = 1
    while i < len(text) and depth:
        if text[i] == "{": depth += 1
        elif text[i] == "}": depth -= 1
        i += 1
    return text[idx + len(key):i - 1].strip() if depth == 0 else None


def safe_grade_answer(given: str, ground_truth: str, timeout: float = 2.0) -> bool:
    """Grade an answer using SymPy if available, else string matching."""
    if HAS_MATH_GRADING:
        try:
            result = run_with_timeout_signal(
                grade_answer,
                args=(given, ground_truth),
                timeout_seconds=int(math.ceil(timeout)),
            )
            if result is not None:
                return result
        except Exception:
            pass
    
    # Fallback: normalize and compare strings
    def normalize(s: str) -> str:
        s = s.strip().replace(" ", "").replace(",", "")
        if s.endswith(".0"):
            s = s[:-2]
        return s.lower()
    
    return normalize(given) == normalize(ground_truth)


# Quick test
print("Answer grading tests:")
print(f"  extract_boxed_answer('... \\\\boxed{{42}}') = {extract_boxed_answer('The answer is \\\\boxed{42}')}")
print(f"  extract_boxed_answer('no box') = {extract_boxed_answer('no box')}")
print(f"  safe_grade_answer('42', '42') = {safe_grade_answer('42', '42')}")
print(f"  safe_grade_answer('43', '42') = {safe_grade_answer('43', '42')}")
print("\u2713 Grading OK")


In [7]:
# ============================================================
# Cell 7: Tool-Use Math Environment (MessageEnv subclass)
# ============================================================
# [FIX] THREE major changes:
# 1. System prompt uses create_conversation_prefix_with_tools() for proper
#    Harmony tool protocol (the model now actually knows HOW to call tools)
# 2. _extract_code_from_message() properly handles parsed ToolCalls from
#    the GptOssRenderer (function.name + function.arguments JSON)
# 3. episode_done=False when model reasons without tool call (multi-turn)

from tinker_cookbook.utils import logtree

# [FIX] Define the Python tool spec for the Harmony protocol
PYTHON_TOOL_SPEC: ToolSpec = {
    "name": "python",
    "description": (
        "Execute Python code. The environment is a stateful Jupyter notebook "
        "with math, numpy (as np), and sympy (as sp) pre-imported. "
        "Always use print() to display results. Code persists between calls."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "Python code to execute"
            }
        },
        "required": ["code"]
    }
}

SYSTEM_PROMPT = (
    "You are a mathematical problem solver with access to a Python tool.\n\n"
    "# Strategy:\n"
    "1. Think through the problem step by step\n"
    "2. Use the python tool to verify computations and explore\n"
    "3. When confident in your answer, put it inside \\boxed{}\n\n"
    "# Rules:\n"
    "- Use print() in your code to see results\n"
    "- You can call the tool multiple times\n"
    "- Put your FINAL answer in \\boxed{} format (e.g., \\boxed{42})\n"
    "- Do NOT just guess — verify with computation when possible"
)


class ToolUseMathEnv(MessageEnv):
    """Multi-turn math environment with Python tool execution.
    
    The model can call a Python tool to compute results, then continue
    reasoning. The episode ends when:
      - The model produces a \\boxed{} answer (graded for reward)
      - Max iterations reached (negative reward)
      - Context overflow (handled by EnvFromMessageEnv wrapper)
    
    Reward:
      +1.0 for correct answer
      -1.0 for wrong answer, no answer, or max iterations
       0.0 for intermediate tool-call steps
    """
    
    def __init__(
        self,
        problem: str,
        answer: str,
        code_executor: CodeExecutor,   # [FIX v2] Was JupyterSession — now lightweight CodeExecutor
        renderer_name: str,
        model_name: str,              # [FIX] Need model name for tokenizer
        python_timeout: float = 30.0,
        max_iterations: int = 15,
    ):
        self.problem = problem
        self.answer = str(answer).strip()
        self.executor = code_executor    # [FIX v2] Was self.jupyter
        self.renderer_name = renderer_name
        self.model_name = model_name
        self.python_timeout = python_timeout
        self.max_iterations = max_iterations
        self.iteration = 0
        self._conversation: list[Message] = []
    
    async def initial_observation(self) -> list[Message]:
        """Build the initial conversation with proper Harmony tool definitions."""
        # [FIX] Use the renderer's create_conversation_prefix_with_tools()
        # to set up the Harmony tool protocol properly. This creates:
        #   1. Internal system msg: "Calls to these tools must go to the commentary channel: 'functions'."
        #   2. Developer msg with: "# Instructions\n\n{system_prompt}\n\n# Tools\n\nnamespace functions { ... }"
        tokenizer = get_tokenizer(self.model_name)
        renderer = renderers.get_renderer(self.renderer_name, tokenizer=tokenizer)
        
        prefix_messages = renderer.create_conversation_prefix_with_tools(
            tools=[PYTHON_TOOL_SPEC],
            system_prompt=SYSTEM_PROMPT,
        )
        
        user_msg: Message = {
            "role": "user",
            "content": (
                self.problem +
                "\n\nPlease reason step by step, use the python tool to verify "
                "your computations, and put your final answer within \\boxed{}."
            ),
        }
        self._conversation = prefix_messages + [user_msg]
        return list(self._conversation)
    
    def _extract_code_from_message(self, message: Message) -> str | None:
        """Extract Python code from an assistant message.
        
        [FIX] The GptOssRenderer parses tool calls into message["tool_calls"]
        as ToolCall objects. When the model calls to=functions.python,
        the parsed ToolCall has:
          - function.name = "python"  
          - function.arguments = '{"code": "print(2+2)"}'  (JSON string)
        
        We extract the code from the JSON arguments.
        Also handles the fallback case where code appears in ```python blocks.
        """
        # [FIX] Primary path: check parsed tool_calls from the renderer
        tool_calls = message.get("tool_calls", [])
        if tool_calls:
            for tc in tool_calls:
                if tc.function.name == "python":
                    try:
                        args = json.loads(tc.function.arguments)
                        code = args.get("code", "")
                        if code.strip():
                            return code
                    except (json.JSONDecodeError, AttributeError):
                        # If JSON parsing fails, try using arguments directly as code
                        if tc.function.arguments.strip():
                            return tc.function.arguments
        
        # [FIX] Also check unparsed_tool_calls — the model might produce
        # malformed JSON but we can still try to extract the code
        unparsed = message.get("unparsed_tool_calls", [])
        if unparsed:
            for utc in unparsed:
                # Try to extract code from raw text
                raw = utc.raw_text
                # Look for code between json markers or just use the raw text
                try:
                    # Sometimes the model sends {"code": "..."} but with extra whitespace
                    parsed = json.loads(raw.strip())
                    if isinstance(parsed, dict) and "code" in parsed:
                        return parsed["code"]
                except (json.JSONDecodeError, TypeError):
                    pass
        
        return None
    
    def _check_for_boxed_answer(self, message: Message) -> str | None:
        """Check if the assistant message contains a \\boxed{} answer."""
        content = message.get("content", "")
        if isinstance(content, list):
            content = " ".join(
                c.get("text", "") if isinstance(c, dict) and c.get("type") == "text"
                else c.get("thinking", "") if isinstance(c, dict) and c.get("type") == "thinking"
                else str(c) if not isinstance(c, dict)
                else ""
                for c in content
            )
        return extract_boxed_answer(content)
    
    async def step(self, message: Message) -> MessageStepResult:
        """Process an assistant message: execute tool or grade answer."""
        self.iteration += 1
        self._conversation.append(message)
        
        # Check for final answer FIRST (model might call tool AND provide answer)
        predicted = self._check_for_boxed_answer(message)
        if predicted is not None:
            correct = safe_grade_answer(predicted, self.answer)
            reward = 1.0 if correct else -1.0
            
            # Log reward info
            with logtree.scope_header("Answer Check"):
                logtree.table_from_dict({
                    "predicted": predicted,
                    "ground_truth": self.answer,
                    "correct": correct,
                    "reward": reward,
                    "iterations": self.iteration,
                }, caption="Final answer")
            
            return MessageStepResult(
                reward=reward,
                episode_done=True,
                next_messages=[],
                metrics={"correct": float(correct), "iterations": self.iteration},
            )
        
        # Check for tool call
        code = self._extract_code_from_message(message)
        if code is not None:
            # Execute Python code via in-process CodeExecutor
            try:
                output = await asyncio.to_thread(
                    self.executor.execute, code, self.python_timeout
                )
            except Exception as e:
                output = f"[ERROR] {type(e).__name__}: {e}"
            
            # [FIX] Build tool response message with proper "name" field
            # The GptOssRenderer requires "name" for tool results:
            # <|start|>functions.python to=assistant<|channel|>commentary<|message|>{output}<|end|>
            tool_response: Message = {
                "role": "tool",
                "name": "python",  # [FIX] Required by GptOssRenderer
                "content": output,
            }
            self._conversation.append(tool_response)
            
            # Check if we've hit max iterations
            if self.iteration >= self.max_iterations:
                return MessageStepResult(
                    reward=-1.0,
                    episode_done=True,
                    next_messages=[],
                    metrics={"correct": 0.0, "max_iterations_hit": 1.0, "iterations": self.iteration},
                )
            
            # Continue the conversation
            return MessageStepResult(
                reward=0.0,  # No intermediate reward
                episode_done=False,
                next_messages=list(self._conversation),
                metrics={},
            )
        
        # No tool call and no boxed answer — model is just reasoning
        # Check if we've hit max iterations
        if self.iteration >= self.max_iterations:
            return MessageStepResult(
                reward=-1.0,
                episode_done=True,
                next_messages=[],
                metrics={"correct": 0.0, "no_answer": 1.0, "iterations": self.iteration},
            )
        
        # [FIX] Let the model continue! It might be in the middle of reasoning
        # and will call a tool or produce \boxed{} on the next turn.
        # Previously: episode_done=True which killed multi-turn reasoning.
        return MessageStepResult(
            reward=0.0,
            episode_done=False,  # [FIX] Was True — that killed multi-turn!
            next_messages=list(self._conversation),  # [FIX] Was [] — need full conversation
            metrics={"no_tool_no_answer": 1.0, "iterations": self.iteration},
        )


print("\u2713 ToolUseMathEnv defined (FIXED)")
print("  Multi-turn: model reasons -> calls python -> gets output -> continues")
print(f"  Max {RLConfig.max_tool_iterations} tool iterations, {RLConfig.python_timeout}s timeout per exec")
print("  Tool definition uses Harmony protocol (create_conversation_prefix_with_tools)")
print("  episode_done=False for reasoning-only turns (allows multi-turn)")

In [8]:
# ============================================================
# Cell 8: EnvGroupBuilder with Tool Execution
# ============================================================
# [FIX v2] Changes:
# 1. Removed frozen=True (was causing FrozenInstanceError)
# 2. Replaced kernel pool with per-env CodeExecutor (zero-cost)
# 3. cleanup() is now a no-op — no resources to release

@dataclass  # [FIX] Removed frozen=True — builder needs mutable state
class ToolMathGroupBuilder(EnvGroupBuilder):
    """Builds a group of ToolUseMathEnv wrapped in EnvFromMessageEnv.
    
    [FIX v2] Each env gets its own CodeExecutor (zero-cost in-process exec).
    No kernel pool, no resource management, no cleanup needed.
    """
    
    problem: str = ""
    answer: str = ""
    num_envs: int = 1
    renderer_name: str = ""
    model_name: str = ""
    python_timeout: float = 30.0
    max_iterations: int = 15
    max_trajectory_tokens: int = 24576
    max_generation_tokens: int = 4096
    
    async def make_envs(self) -> Sequence[Env]:
        """Create num_envs ToolUseMathEnv instances, each with its own CodeExecutor."""
        tokenizer = get_tokenizer(self.model_name)
        renderer = renderers.get_renderer(self.renderer_name, tokenizer=tokenizer)
        
        envs = []
        
        for _ in range(self.num_envs):
            # [FIX v2] Create a lightweight CodeExecutor per env — zero cost!
            # No pool, no waiting, no resource limits.
            executor = CodeExecutor(timeout=self.python_timeout)
            
            msg_env = ToolUseMathEnv(
                problem=self.problem,
                answer=self.answer,
                code_executor=executor,
                renderer_name=self.renderer_name,
                model_name=self.model_name,
                python_timeout=self.python_timeout,
                max_iterations=self.max_iterations,
            )
            
            env = EnvFromMessageEnv(
                renderer=renderer,
                message_env=msg_env,
                failed_parse_reward=-1.0,
                terminate_on_parse_error=True,
                max_trajectory_tokens=self.max_trajectory_tokens,
                max_generation_tokens=self.max_generation_tokens,
                context_overflow_reward=-0.1,
            )
            envs.append(env)
        
        return envs
    
    async def compute_group_rewards(
        self, trajectory_group: list[Trajectory], env_group: Sequence[Env]
    ) -> list[tuple[float, Metrics]]:
        """No additional group-level rewards (all rewards come from step)."""
        return [(0.0, {}) for _ in trajectory_group]
    
    async def cleanup(self) -> None:
        """No-op. CodeExecutor holds no external resources to cleanup."""
        pass
    
    def logging_tags(self) -> list[str]:
        return ["aimo_tool"]


print("\u2713 ToolMathGroupBuilder defined (FIXED v2)")
print(f"  Each group: {RLConfig.group_size} envs, each with its own CodeExecutor")
print(f"  No kernel pool — unlimited concurrency, zero resource cost")
print(f"  Max trajectory tokens: {RLConfig.max_trajectory_tokens}")
print(f"  Max generation tokens per turn: {RLConfig.max_tokens}")

In [9]:
# ============================================================
# Cell 9: Custom RLDataset & RLDatasetBuilder (UNCHANGED except constructor args)
# ============================================================

class AIMOToolDataset(RLDataset):
    """RL dataset wrapping our CSV with tool-use environment builders."""
    
    def __init__(
        self,
        problems: list[dict],
        batch_size: int,
        group_size: int,
        renderer_name: str,
        model_name: str,
        python_timeout: float = 30.0,
        max_iterations: int = 15,
        max_trajectory_tokens: int = 24576,
        max_generation_tokens: int = 4096,
    ):
        self.problems = problems
        self.batch_size = batch_size
        self.group_size = group_size
        self.renderer_name = renderer_name
        self.model_name = model_name
        self.python_timeout = python_timeout
        self.max_iterations = max_iterations
        self.max_trajectory_tokens = max_trajectory_tokens
        self.max_generation_tokens = max_generation_tokens
    
    def get_batch(self, index: int) -> Sequence[EnvGroupBuilder]:
        batch_start = (index * self.batch_size) % len(self.problems)
        indices = []
        for i in range(self.batch_size):
            indices.append((batch_start + i) % len(self.problems))
        
        builders = []
        for i in indices:
            p = self.problems[i]
            builder = ToolMathGroupBuilder(
                problem=p["problem"],
                answer=str(p["answer"]),
                num_envs=self.group_size,
                renderer_name=self.renderer_name,
                model_name=self.model_name,
                python_timeout=self.python_timeout,
                max_iterations=self.max_iterations,
                max_trajectory_tokens=self.max_trajectory_tokens,
                max_generation_tokens=self.max_generation_tokens,
            )
            builders.append(builder)
        return builders
    
    def __len__(self) -> int:
        return max(1000, math.ceil(len(self.problems) / self.batch_size))


@chz.chz
class AIMOToolDatasetBuilder(RLDatasetBuilder):
    """Builds train (and optionally test) datasets from our CSV."""
    
    dataset_path: str
    batch_size: int
    group_size: int
    model_name_for_tokenizer: str
    renderer_name: str
    python_timeout: float = 30.0
    max_iterations: int = 15
    max_trajectory_tokens: int = 24576
    max_generation_tokens: int = 4096
    eval_split: int = 5
    seed: int = 42
    
    async def __call__(self) -> tuple[AIMOToolDataset, AIMOToolDataset | None]:
        import random
        
        df = pd.read_csv(self.dataset_path)
        all_problems = df.to_dict('records')
        
        rng = random.Random(self.seed)
        rng.shuffle(all_problems)
        
        eval_problems = all_problems[:self.eval_split]
        train_problems = all_problems[self.eval_split:]
        
        logger.info(f"Dataset: {len(train_problems)} train, {len(eval_problems)} eval")
        
        # [FIX v2] No kernel pool to initialize — CodeExecutor is zero-cost
        
        train_dataset = AIMOToolDataset(
            problems=train_problems,
            batch_size=self.batch_size,
            group_size=self.group_size,
            renderer_name=self.renderer_name,
            model_name=self.model_name_for_tokenizer,
            python_timeout=self.python_timeout,
            max_iterations=self.max_iterations,
            max_trajectory_tokens=self.max_trajectory_tokens,
            max_generation_tokens=self.max_generation_tokens,
        )
        
        eval_dataset = AIMOToolDataset(
            problems=eval_problems,
            batch_size=len(eval_problems),
            group_size=1,
            renderer_name=self.renderer_name,
            model_name=self.model_name_for_tokenizer,
            python_timeout=self.python_timeout,
            max_iterations=self.max_iterations,
            max_trajectory_tokens=self.max_trajectory_tokens,
            max_generation_tokens=self.max_generation_tokens,
        )
        
        return train_dataset, eval_dataset


print(f"\u2713 AIMOToolDataset and AIMOToolDatasetBuilder defined")
print(f"  {len(df)} problems: {len(df) - 5} train + 5 eval")
print(f"  Each step: {RLConfig.batch_size} problems x {RLConfig.group_size} completions = {RLConfig.batch_size * RLConfig.group_size} rollouts")
print(f"  Each rollout: up to {RLConfig.max_tool_iterations} tool calls")
print(f"  Executor: in-process CodeExecutor (no kernel pool needed)")


In [10]:
# ============================================================
# Cell 10: Configure & Launch RL Training (UNCHANGED)
# ============================================================

renderer_name = model_info.get_recommended_renderer_name(RLConfig.model_name)
print(f"Using renderer: {renderer_name}")

dataset_builder = AIMOToolDatasetBuilder(
    dataset_path=RLConfig.dataset_path,
    batch_size=RLConfig.batch_size,
    group_size=RLConfig.group_size,
    model_name_for_tokenizer=RLConfig.model_name,
    renderer_name=renderer_name,
    python_timeout=RLConfig.python_timeout,
    max_iterations=RLConfig.max_tool_iterations,
    max_trajectory_tokens=RLConfig.max_trajectory_tokens,
    max_generation_tokens=RLConfig.max_tokens,
    eval_split=1,
    seed=42,
)

config = train.Config(
    # Core
    model_name=RLConfig.model_name,
    learning_rate=RLConfig.learning_rate,
    dataset_builder=dataset_builder,
    max_tokens=RLConfig.max_tokens,
    log_path=RLConfig.log_path,
    
    # LoRA
    lora_rank=RLConfig.lora_rank,
    
    # Loss
    loss_fn=RLConfig.loss_fn,
    
    # Sampling
    temperature=RLConfig.temperature,
    
    # Checkpointing & eval
    eval_every=RLConfig.eval_every,
    save_every=RLConfig.save_every,
    max_steps=RLConfig.max_steps,
    
    # Renderer
    renderer_name=renderer_name,
    
    # Optional: warm-start from SFT
    load_checkpoint_path=RLConfig.load_checkpoint_path,
    
    # Error tolerance for tool execution flakes
    rollout_error_tolerance=True,  # Retry on failure (max 3 retries)
    
    # Logging
    num_groups_to_log=2,
)

print("\u2554" + "\u2550"*60 + "\u2557")
print("\u2551  GPT OSS 120B RLVR + Tool-Integrated Reasoning")
print("\u2560" + "\u2550"*60 + "\u2563")
print(f"\u2551  Model:          {config.model_name}")
print(f"\u2551  LoRA rank:      {config.lora_rank}")
print(f"\u2551  Learning rate:  {config.learning_rate:.1e}")
print(f"\u2551  Max tokens/turn:{RLConfig.max_tokens}")
print(f"\u2551  Max trajectory: {RLConfig.max_trajectory_tokens} tokens")
print(f"\u2551  Batch size:     {RLConfig.batch_size} problems")
print(f"\u2551  Group size:     {RLConfig.group_size} completions/problem")
print(f"\u2551  Rollouts/step:  {RLConfig.batch_size * RLConfig.group_size}")
print(f"\u2551  Executor:       In-process CodeExecutor (no pool needed)")
print(f"\u2551  Tool iters:     Up to {RLConfig.max_tool_iterations} per episode")
print(f"\u2551  Python timeout: {RLConfig.python_timeout}s")
print(f"\u2551  Max steps:      {config.max_steps}")
print(f"\u2551  Loss fn:        {config.loss_fn}")
print(f"\u2551  Renderer:       {config.renderer_name}")
print(f"\u2551  Error tolerance: {config.rollout_error_tolerance}")
print(f"\u2551  Checkpoint:     {config.load_checkpoint_path or 'None (base model)'}")
print("\u255a" + "\u2550"*60 + "\u255d")
print("\n\u23f3 Launching RLVR training with tool use...\n")

# Run training
# asyncio.run(train.main(config))

# In Jupyter, there's already a running event loop, so use await directly
await train.main(config)

2026-04-11 06:38:39,033 [INFO] 
Configuration:
  learning_rate: 1e-05
  dataset_builder: {'dataset_path': '/kaggle/input/datasets/nahidhossainredom/rlvr-dataset/rlvr_dataset.csv', 
'batch_size': 4, 'group_size': 8, 'mo ... imeout': 30.0, 'max_iterations': 15, 'max_trajectory_tokens': 24576, 
'max_generation_tokens': 4096, 'eval_split': 1, 'seed': 42}
  model_name: 'openai/gpt-oss-120b'
  max_tokens: 4096
  log_path: '/kaggle/working/rl_logs'
  eval_every: 20
  save_every: 25
  evaluator_builders: []
  load_checkpoint_path: None
  renderer_name: 'gpt_oss_no_sysprompt'
  wandb_project: None
  wandb_name: None
  kl_penalty_coef: 0.0
  kl_discount_factor: 0.0
  kl_reference_config: None
  loss_fn: 'importance_sampling'
  loss_fn_config: None
  num_substeps: 1
  lora_rank: 32
  temperature: 1.0
  compute_post_kl: False
  remove_constant_reward_groups: False
  rollout_error_tolerance: True
  enable_trace: False
  span_chart_every: 0
  async_config: None
  stream_minibatch_config: None
  base

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

huggingface_hub.utils._http:779 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

gpt-oss-rlvr-tool:85 [INFO] Dataset: 28 train, 1 eval
tinker_cookbook.rl.train:1959 [INFO] Will train on 100 batches
Sampling batch 0: 100%|██████████| 4/4 [08:08<00:00, 122.06s/it]
tinker_cookbook.rl.train:259 [INFO] 
====== Trajectory Group ======
****** trajectory idx=0, reward=-1 ******
Per-step metrics:
  Step 0:
    no_tool_no_answer: 1.0
    iterations: 1
  Step 1:
    no_tool_no_answer: 1.0
    iterations: 3
  Step 2:
    no_tool_no_answer: 1.0
    iterations: 4
  Step 3:
    correct: 0.0
    iterations: 6
---- datum ----
<|start|>system<|message|>Calls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>developer<|message|># Instructions

You are a mathematical problem solver with access to a Python tool.

# Strategy:
1. Think through the problem step by step
2. Use the python tool to verify computations and explore
3. When confident in your answer, put it inside \boxed{}

# Rules:
- Use print() in your code to see results
- You can call the tool mult

tinker_cookbook.utils.ml_log:279 [INFO] 
                                 Step 0                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Metric                                                  ┃ Value       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ env/all/ac_tokens_per_turn                              │ 666.272374  │
│ env/all/by_group/frac_all_bad                           │ 0.250000    │
│ env/all/by_group/frac_all_good                          │ 0.000000    │
│ env/all/by_group/frac_mixed                             │ 0.750000    │
│ env/all/correct                                         │ 0.387097    │
│ env/all/iterations                                      │ 4.298246    │
│ env/all/max_iterations_hit                              │ 1.000000    │
│ env/all/max_tokens_reached                              │ 1.000000    │
│ env/all/no_tool_no_answer                               │ 1.000000   

tinker_cookbook.utils.ml_log:279 [INFO] 
                                 Step 1                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Metric                                                  ┃ Value       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ env/all/ac_tokens_per_turn                              │ 573.165242  │
│ env/all/by_group/frac_all_bad                           │ 0.250000    │
│ env/all/by_group/frac_all_good                          │ 0.000000    │
│ env/all/by_group/frac_mixed                             │ 0.750000    │
│ env/all/correct                                         │ 0.225806    │
│ env/all/iterations                                      │ 5.074830    │
│ env/all/max_iterations_hit                              │ 1.000000    │
│ env/all/max_tokens_reached                              │ 1.000000    │
│ env/all/no_tool_no_answer                               │ 1.000000   

tinker_cookbook.utils.ml_log:279 [INFO] 
                                 Step 2                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Metric                                                  ┃ Value       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ env/all/ac_tokens_per_turn                              │ 586.901186  │
│ env/all/by_group/frac_all_bad                           │ 0.000000    │
│ env/all/by_group/frac_all_good                          │ 0.000000    │
│ env/all/by_group/frac_mixed                             │ 1.000000    │
│ env/all/correct                                         │ 0.166667    │
│ env/all/iterations                                      │ 4.945312    │
│ env/all/max_iterations_hit                              │ 1.000000    │
│ env/all/max_tokens_reached                              │ 1.000000    │
│ env/all/no_answer                                       │ 1.000000   

In [ ]:
# ============================================================
# Cell 11: Download Trained Weights
# ============================================================

import requests
import json
from pathlib import Path

def find_final_checkpoint(log_path: str) -> str | None:
    """Find the final checkpoint path from the training logs."""
    checkpoints_file = Path(log_path) / "checkpoints.jsonl"
    if not checkpoints_file.exists():
        print(f"No checkpoints file found at {checkpoints_file}")
        return None
    
    last_checkpoint = None
    with open(checkpoints_file) as f:
        for line in f:
            try:
                data = json.loads(line.strip())
                if 'sampler_path' in data:
                    last_checkpoint = data['sampler_path']
            except json.JSONDecodeError:
                continue
    return last_checkpoint


def download_weights(sampler_path: str, output_dir: str = "gpt_oss_120b_rlvr_tool_weights"):
    """Download the trained LoRA weights from Tinker."""
    os.makedirs(output_dir, exist_ok=True)
    
    service_client = tinker.ServiceClient()
    rest_client = service_client.create_rest_client()
    url_resp = rest_client.get_checkpoint_archive_url_from_tinker_path(sampler_path).result()
    
    print(f"Downloading checkpoint from: {sampler_path}")
    r = requests.get(url_resp.url, stream=True)
    r.raise_for_status()
    total_bytes = int(r.headers.get('content-length', 0))
    print(f"  File size: {total_bytes / 1e9:.2f} GB")
    
    output_file = os.path.join(output_dir, "lora_checkpoint.tar")
    downloaded = 0
    with open(output_file, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total_bytes:
                pct = 100 * downloaded / total_bytes
                print(f"\r  Progress: {pct:.1f}% ({downloaded/1e9:.2f}/{total_bytes/1e9:.2f} GB)", end="")
    
    print(f"\n  \u2713 Saved: {output_file} ({downloaded / 1e9:.2f} GB)")
    return output_file


# Find and download the final checkpoint
sampler_path = find_final_checkpoint(RLConfig.log_path)
if sampler_path:
    print(f"Found checkpoint: {sampler_path}")
    download_weights(sampler_path)
else:
    print("No checkpoint found. Training may not have completed.")